In [7]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [8]:
# Carrega as planilhas
df1 = pd.read_excel("../Datos_Anonimo_20231_v2.xlsx", sheet_name=1)
df2 = pd.read_excel("../Datos_Anonimo_20232_v2.xlsx", sheet_name=1)

In [9]:
# Mapeamentos
mapa_cols = {
    "Periodo": "Periodo",
    "Grupo": "Grupo",
    "Horario": "Horario",
    "Tipo_Documento": "Tipo_Documento",
    "Edad": "Idade",
    "Genero": "Genero",
    "Nombre_Programa_Academico": "Nome_Programa_Acadêmico",
    "STEM": "STEM",
    "Parcial_1": "Parcial_1",
    "Parcial_2": "Parcial_2",
    "Proyecto_Parte1": "Projeto_Parte1",
    "Proyecto_Parte2": "Projeto_Parte2",
    "Talleres": "Oficinas",
    "Quices": "Quizzes",
    "Fecha_Quiz1": "Data_Quiz1",
    "Quiz1": "Quiz1",
    "TiempoQ1": "TempoQ1",
    "Fecha_Quiz2": "Data_Quiz2",
    "Quiz2": "Quiz2",
    "TiempoQ2": "TempoQ2",
    "Fecha_Quiz3": "Data_Quiz3",
    "Quiz3": "Quiz3",
    "TiempoQ3": "TempoQ3",
    "Fecha_Quiz4": "Data_Quiz4",
    "Quiz4": "Quiz4",
    "TiempoQ4": "TempoQ4",
    "Fecha_Quiz5": "Data_Quiz5",
    "Quiz5": "Quiz5",
    "TiempoQ5": "TempoQ5",
    "Fecha_Quiz6": "Data_Quiz6",
    "Quiz6": "Quiz6",
    "TiempoQ6": "TempoQ6",
    "Fecha_Quiz7": "Data_Quiz7",
    "Quiz7": "Quiz7",
    "TiempoQ7": "TempoQ7",
    "Quiz8": "Quiz8",
    "CalcNotaQuiz": "CalcNotaQuiz",
    "MejoraNotaQuices": "MelhoraNotaQuizzes",
    "Cuánto mejora?": "Quanto_melhora?",
    "Calificación_Oficial": "Nota_Oficial",
    "Aprobo": "Aprovou"
}
mapa_programas = {
    "COMUNICACIÓN SOCIAL": "Comunicacao Social",
    "DERECHO": "Direito",
    "INGENIERÍA CIVIL": "Engenharia Civil",
    "ADMINISTRACIÓN DE NEGOCIOS": "Administração de Negocios",
    "INGENIERÍA DE DISEÑO DE PRODUCTO": "Engenharia de Design de Produto",
    "MERCADEO": "Marketing",
    "PSICOLOGÍA": "Psicologia",
    "INGENIERÍA FÍSICA": "Engenharia Fisica",
    "NEGOCIOS INTERNACIONALES": "Negocios Internacionais",
    "BIOLOGÍA": "Biologia",
    "CIENCIAS POLÍTICAS": "Ciencias Politicas",
    "ECONOMÍA": "Economia",
    "CONTADURÍA PÚBLICA": "Contabilidade Publica",
    "MÚSICA": "Musica",
    "LITERATURA": "Literatura",
    "INGENIERÍA DE PROCESOS": "Engenharia de Processos",
    "CONVENIO MOVILIDAD PREGRADO (CONVENIOS - MOVILIDAD NACIONAL - ASISTENTES PREGRADO)":
        "Convenio Mobilidade Graduacao (Convenios - Mobilidade Nacional - Assistentes Graduacao)",
}
mapa_aprov = {
    "Aprobó": "Aprovou",
    "Reprobó": "Reprovou"
}
mapa_idade = {
    "Mayor": "Maior"
}
mapa_gen = {
    "femenino": "Feminino",
    "masculino": "Masculino"
}

In [10]:
def padroniza_df(df):
    df = df.rename(columns=mapa_cols)
    col_prog = "Nome_Programa_Academico"
    if col_prog in df.columns:
        df[col_prog] = (
            df[col_prog]
            .astype(str)
            .str.strip()
            .replace(mapa_programas, regex=False)
        )
    col_aprov = "Aprovou"
    if col_aprov in df.columns:
        df[col_aprov] = (
            df[col_aprov]
            .astype(str)
            .str.strip()
            .replace(mapa_aprov, regex=False)
        )
    col_idade = "Idade"
    if col_idade in df.columns:
        df[col_idade] = (
            df[col_idade]
            .astype(str)
            .str.strip()
            .replace(mapa_idade, regex=False)
        )
    col_gen = "Genero"
    if col_gen in df.columns:
        df[col_gen] = (
            df[col_gen]
            .astype(str)
            .str.strip()
            .replace(mapa_gen, regex=False)
        )
    return df

df1 = padroniza_df(df1)
df2 = padroniza_df(df2)

In [11]:
cat_cols = ['Tipo_Documento', 'Idade', 'Genero', 'STEM', 'MelhoraNotaQuizzes', 'Aprovou']
df_cat1 = df1[cat_cols].dropna(subset=['STEM'])
df_cat2 = df2[cat_cols].dropna(subset=['STEM'])

In [ ]:
# Fit o encoder no conjunto combinado para garantir as mesmas colunas
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(pd.concat([df_cat1, df_cat2], axis=0))

# Transforma os dois DataFrames
encoded1 = encoder.transform(df_cat1)
encoded2 = encoder.transform(df_cat2)

encoded_df1 = pd.DataFrame(encoded1, columns=encoder.get_feature_names_out(cat_cols), index=df_cat1.index)
encoded_df2 = pd.DataFrame(encoded2, columns=encoder.get_feature_names_out(cat_cols), index=df_cat2.index)

# Junta com os DataFrames originais (removendo as colunas categóricas)
df_cat1_encoded = df_cat1.drop(columns=cat_cols).join(encoded_df1)
df_cat2_encoded = df_cat2.drop(columns=cat_cols).join(encoded_df2)

# Junta os dois DataFrames finais
df_new_categorical = pd.concat([df_cat1_encoded, df_cat2_encoded], axis=0, ignore_index=True)

df_new_categorical = df_new_categorical.loc[:, ~df_new_categorical.columns.duplicated()]

df_new_categorical